# Geometric & Intensity Transformations


Apply classical image processing operations using Python and the `(PIL)` library. Complete both tasks and save your output images for comparison.


In [ ]:
# Import the library
import glob
import numpy as np
from PIL import Image, ImageOps

In [ ]:
# ── Load image ────────────────────────────────────
# glob is used so the code works whatever the extension is (.jpg / .png / .jpeg)
import os
folder = os.path.expanduser(r'~\Downloads\LAB3 IMAGE')
image_path = glob.glob(os.path.join(folder, 'nature.*'))[0]
os.chdir(folder)

img = Image.open(image_path).convert('RGB')
img

### 1. Scale (increase size)
Double the image dimensions using `Image.resize()` with high-quality resampling `(LANCZOS)`.

In [ ]:
# Get the size of the image
width, height = img.size

# scaling factors
cx, cy = 2, 2

In [ ]:
# ── 1. Increase size (scale ×2) ────────────────────
new_width, new_height = int(width * cx), int(height * cy)

# Resize the image using PIL's built-in method
scaled_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
scaled_img

In [ ]:

# Save the scaled image and print the sizes (The new image name should be "task1_1_scaled.jpg")
scaled_img.save('task1_1_scaled.jpg')

print('Original size:', img.size)
print('Scaled size  :', scaled_img.size)

 non-uniform scale (cx=2, cy=1) → stretch horizontally only

In [ ]:
# non-uniform scaling factors
cx, cy = 2, 1

In [ ]:
stretch_width, stretch_height = int(width * cx), int(height * cy)

stretched_img = img.resize((stretch_width, stretch_height), Image.Resampling.LANCZOS)
stretched_img

In [ ]:
stretched_img.save('task1_1_stretched.jpg')

print('Original size :', img.size)
print('Stretched size:', stretched_img.size)

### 2. Rotate 120°
Rotate the image by 120 degrees, expanding the canvas to fit the full rotated image.

In [ ]:
# ── 2. Rotate 120 degrees ──────────────────────────
# Save the scaled image and print the sizes (The new image name should be "task1_2_rotated.jpg")
rotated_img = img.rotate(120, resample=Image.Resampling.BICUBIC, expand=True)
rotated_img.save('task1_2_rotated.jpg')

print('Original size:', img.size)
print('Rotated size :', rotated_img.size)

rotated_img

### 3. Shear

In [ ]:
# -- c. Get the image dimensions ────────────────────
width, height = img.size

In [ ]:
# -- d. define the shear matrix ──────────────────────────
# Choose X-axis or Y-axis shear. The shear factor controls how much the image slants — start with 0.5 then experiment.
shear_factor = 0.5

# X-axis shear
shear_matrix = (1, -shear_factor, 0,
                0, 1, 0)

In [ ]:
# -- e. Apply the shear transformation to the image ──────────────────────────
# PIL's transform() takes the inverse affine matrix:
# [ 1    shx   tx ]
# [ shy  1     ty ]
sheared_width = width + int(shear_factor * height)

sheared_img = img.transform((sheared_width, height), Image.Transform.AFFINE, shear_matrix,
                            resample=Image.Resampling.BICUBIC, fillcolor=(255, 255, 255))
sheared_img

In [ ]:
# -- f. Save the sheared image(The new image name should be "task1_3_sheared.jpg") 
sheared_img.save('task1_3_sheared.jpg')

print('Original size:', img.size)
print('Sheared size :', sheared_img.size)

### Experiment and compare
Try these:

— Change shear factor from 0.5 to 0.1, 0.3, 0.8 and compare results

— Switch from X-axis to Y-axis shear matrix

— Update the canvas multiplier to match your new factor

— Try combining X and Y shear in one matrix

In [ ]:
# ── Experiment and compare ─────────────────────────
# a. different X-axis shear factors
for f in [0.1, 0.3, 0.8]:
    matrix = (1, -f, 0,
              0, 1, 0)
    out = img.transform((width + int(f * height), height), Image.Transform.AFFINE, matrix,
                        resample=Image.Resampling.BICUBIC, fillcolor=(255, 255, 255))
    out.save(f'task1_3_shear_x_{f}.jpg')
    print('X shear factor', f, '-> size', out.size)
    display(out)

# b. Y-axis shear
y_matrix = (1, 0, 0,
            -0.5, 1, 0)
y_sheared_img = img.transform((width, height + int(0.5 * width)), Image.Transform.AFFINE, y_matrix,
                              resample=Image.Resampling.BICUBIC, fillcolor=(255, 255, 255))
y_sheared_img.save('task1_3_sheared_y.jpg')
print('Y shear -> size', y_sheared_img.size)
display(y_sheared_img)

# c. combined X and Y shear
shx, shy = 0.3, 0.3
xy_matrix = (1, -shx, 0,
             -shy, 1, 0)
xy_sheared_img = img.transform((width + int(shx * height), height + int(shy * width)),
                               Image.Transform.AFFINE, xy_matrix,
                               resample=Image.Resampling.BICUBIC, fillcolor=(255, 255, 255))
xy_sheared_img.save('task1_3_sheared_xy.jpg')
print('XY shear -> size', xy_sheared_img.size)
display(xy_sheared_img)

# Intensity Transformations
Negative · Log · Power Law (Gamma)

In [ ]:

# ── 1. Negative ──────────────────────────────────── 
# Method 1: NumPy array manipulation
arr = np.array(img)

negative_arr = 255 - arr
negative_img = Image.fromarray(negative_arr)
negative_img.save('task2_1_negative.jpg')
negative_img

In [ ]:
# Method 2: PIL's ImageOps
negative_img_2 = ImageOps.invert(img)
negative_img_2

In [ ]:
# ── 2. Log transformation ──────────────────────────

# s = c · log(1 + r) --> We need to find c.
# We want the maximum output value (s) to be 255 when the maximum input value (r) is 255:
# 255 = c · log(1 + 255)
# c = 255 / log(1 + 255)
# float32 is used to stop 1 + 255 from overflowing back to 0 in uint8
arr = np.array(img, dtype=np.float32)

c = 255 / np.log(1 + 255)

# Apply the log transformation to each pixel
log_transformed_arr = c * np.log(1 + arr)
log_img = Image.fromarray(np.uint8(log_transformed_arr))
log_img.save('task2_2_log.jpg')
log_img

In [ ]:

# ── 3. Power-law / Gamma correction ───────────────
# s = c · r^gamma  --> gamma < 1 brightens, gamma > 1 darkens
gamma = 0.5

normalized_arr = np.array(img, dtype=np.float32) / 255.0
gamma_arr = 255 * (normalized_arr ** gamma)
gamma_img = Image.fromarray(np.uint8(gamma_arr))
gamma_img.save('task2_3_gamma.jpg')
gamma_img